In [8]:
import polars as pl
import numpy as np
import pandas as pd
import gc
import os
import warnings
warnings.filterwarnings('ignore')

import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, train_test_split
from scipy.optimize import minimize

print(f'LightGBM: {lgb.__version__}')

LightGBM: 4.6.0


In [ ]:
DATA_DIR = 'data/'           
OUT_DIR  = 'data/'
SEED     = 42

SUBSAMPLE_SIZE = 200_000      
TOP_K_FEATURES = 400          

SELECTED_DIR = f'{DATA_DIR}selected_features'
os.makedirs(SELECTED_DIR, exist_ok=True)

BASE_PARAMS = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'n_jobs': -1,
    'verbose': -1,
    'device': 'gpu',
    'gpu_platform_id': 0,
    'gpu_device_id': 0,
}

# 1. Загрузка данных

### Последовательность действий
- загружаем main
- загружаем extra
- объединяем в 1 одну таблицу
- удаляем дубликаты

### Логика дейтсвий
Подготовительный этап в результате которого объединили ~2200 признаки в один DataFrame, снизили разрядность до Float16 для экономии памяти и ускорения, удалили возможные дубликаты

In [ ]:
WIDE_TRAIN = f'{DATA_DIR}train_wide_features.parquet'
WIDE_TEST = f'{DATA_DIR}test_wide_features.parquet'

if not os.path.exists(WIDE_TRAIN):
    print("Создаём wide-таблицы из main + extra...")
    
    train_customer_ids = pl.read_parquet(f'{DATA_DIR}train_main_features.parquet').select('customer_id')
    test_customer_ids = pl.read_parquet(f'{DATA_DIR}test_main_features.parquet').select('customer_id')
    
    lf_main_train = pl.scan_parquet(f'{DATA_DIR}train_main_features.parquet').drop('customer_id')
    lf_extra_train = pl.scan_parquet(f'{DATA_DIR}train_extra_features.parquet').drop('customer_id')
    lf_main_test = pl.scan_parquet(f'{DATA_DIR}test_main_features.parquet').drop('customer_id')
    lf_extra_test = pl.scan_parquet(f'{DATA_DIR}test_extra_features.parquet').drop('customer_id')
    
    train_wide = pl.concat([lf_main_train, lf_extra_train], how='horizontal') \
                   .with_columns(pl.all().cast(pl.Float16)).collect()
    test_wide = pl.concat([lf_main_test, lf_extra_test], how='horizontal') \
                  .with_columns(pl.all().cast(pl.Float16)).collect()
    
    def dedup(df):
        seen = set()
        cols = []
        for c in df.columns:
            if c not in seen:
                seen.add(c)
                cols.append(c)
        return df.select(cols)
    
    train_wide = dedup(train_wide)
    test_wide = dedup(test_wide)
    
    train_wide.write_parquet(WIDE_TRAIN)
    test_wide.write_parquet(WIDE_TEST)
    print("Wide-файлы сохранены.")
else:
    print("Загружаем готовые wide-файлы...")
    train_wide = pl.read_parquet(WIDE_TRAIN)
    test_wide = pl.read_parquet(WIDE_TEST)
    train_customer_ids = pl.read_parquet(f'{DATA_DIR}train_main_features.parquet').select('customer_id')
    test_customer_ids = pl.read_parquet(f'{DATA_DIR}test_main_features.parquet').select('customer_id')

Загружаем готовые wide-файлы...


# 2. Фичеинжиниринг

### Последовательность действий
- отбираем все числовые признаки
- добавляем для каждого клиента 4 признака
    - непустое количество числовых признаков
    - среднее
    - максимальное
    - минимальное


### Логика дейтсвий
Для каждого клиента добавили 4 метапризнака - добавляет информацию об общем паттерне поведения, а не о конкретных значениях.

In [ ]:
def feature_engineering_fast(df: pl.DataFrame) -> pl.DataFrame:
    num_cols = [c for c in df.columns if c.startswith('num_feature')]
    
    if not num_cols:
        return df
    
    return df.with_columns([
        pl.sum_horizontal(
            *(pl.col(c).is_not_null().cast(pl.Int32) for c in num_cols)
        ).alias('num_not_null_count'),
        pl.mean_horizontal(*[pl.col(c) for c in num_cols]).alias('num_row_mean'),
        pl.max_horizontal(*[pl.col(c) for c in num_cols]).alias('num_row_max'),
        pl.min_horizontal(*[pl.col(c) for c in num_cols]).alias('num_row_min'),
    ])

print("Feature engineering...")
train_wide = feature_engineering_fast(train_wide)
test_wide = feature_engineering_fast(test_wide)


Feature engineering...


# 3. Отбор топ-K признаков для каждого таргета

### Логика дейтсвий
Подготовили данные к обучению. Главная идея — персонализированный отбор признаков для каждого продукта. Убрали лишний шум

In [ ]:
target = pl.read_parquet(f'{DATA_DIR}train_target.parquet')
# имена столбцов
target_cols = [c for c in target.columns if c.startswith('target_')]
# имена категориальных признаков
cat_feature_names = [c for c in train_wide.columns if c.startswith('cat_feature')]

# LightGBM работает с 32
train_wide = train_wide.with_columns(pl.col(cat_feature_names).cast(pl.Int32))
test_wide = test_wide.with_columns(pl.col(cat_feature_names).cast(pl.Int32))

def select_features_for_target(target_name, X, y, cat_features, k=TOP_K_FEATURES, subsample_size=SUBSAMPLE_SIZE):
    """Отбирает k наиболее важных признаков с помощью LightGBM"""
    path = f'{SELECTED_DIR}/{target_name}.npy'
    if os.path.exists(path):
        return np.load(path).tolist()
    
    print(f'  Отбор признаков для {target_name}...')
    
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]
    n_pos = len(pos_idx)
    
    if n_pos < subsample_size / 2:
        # крайне редкий класс
        n_neg = min(subsample_size - n_pos, len(neg_idx))
        if n_neg > 0:
            # дополняем отрицательными
            idx = np.concatenate([pos_idx, np.random.choice(neg_idx, n_neg, replace=False)])
        else:
            idx = pos_idx
        # гарантирует что в обучающей выборке будут положительные, хоть и возможно с сильным дисбалансом
    else:
        # если не слишком редкий делаем сбалансированную выборку
        n_pos_sample = min(subsample_size // 2, n_pos)
        n_neg_sample = min(subsample_size // 2, len(neg_idx))
        pos_sample = np.random.choice(pos_idx, n_pos_sample, replace=False)
        neg_sample = np.random.choice(neg_idx, n_neg_sample, replace=False)
        idx = np.concatenate([pos_sample, neg_sample])
    
    idx = np.sort(idx)
    X_sub = X.iloc[idx]
    y_sub = y[idx]
    
    print(f'    Подвыборка: {len(X_sub)} строк (положительных: {y_sub.sum()})')
    
    model = lgb.LGBMClassifier(
        n_estimators=150,     
        learning_rate=0.1, 
        num_leaves=31,
        min_child_samples=10,
        random_state=SEED, 
        n_jobs=-1, 
        verbose=-1
    )
    
    cat_features_subset = [c for c in cat_features if c in X_sub.columns]
    model.fit(X_sub, y_sub, categorical_feature=cat_features_subset)
    
    # в LightGBM по умолчанию важность считается как общее количество раз, когда признак использовался в разбиениях 
    importance = model.feature_importances_
    top_idx = np.argsort(importance)[::-1][:k]
    selected = X_sub.columns[top_idx].tolist()
    
    np.save(path, np.array(selected))
    print(f'    Отобрано {len(selected)} признаков')
    return selected

print("Конвертация в pandas...")
X_train = train_wide.to_pandas()
X_test = test_wide.to_pandas()
y_train = target.select(target_cols).to_pandas()
test_customer_ids_list = test_customer_ids['customer_id'].to_list()

print(f"Размер данных: X_train={X_train.shape}, X_test={X_test.shape}")

selected_features_dict = {}
print("\nОтбор признаков для каждого таргета:")
for i, t in enumerate(target_cols):
    print(f"\n[{i+1}/{len(target_cols)}] {t}...")
    y_t = y_train[t].values
    
    if y_t.sum() == 0:
        print(f'  ⚠️ Нет положительных примеров, берем все признаки')
        selected_features_dict[t] = X_train.columns.tolist()[:TOP_K_FEATURES]
        continue
    
    pos_ratio = y_t.sum() / len(y_t)
    if pos_ratio < 0.01:
        # для продуктов с низкой долей положительных
        # дополнительно уменьшаем размер подвыборки для обучения LightGBM при отборе признаков
        # Умножение на 30 гарантирует, что мы не возьмём больше примеров, чем есть в реальности, и сохраним пропорцию 
        subsample_size = min(SUBSAMPLE_SIZE, len(y_t) * 30)
        print(f'    Редкий класс (ratio={pos_ratio:.4f}), subsample_size={subsample_size}')
    else:
        subsample_size = SUBSAMPLE_SIZE
    
    selected_features_dict[t] = select_features_for_target(
        t, X_train, y_t, cat_feature_names, 
        k=TOP_K_FEATURES, 
        subsample_size=subsample_size
    )

Конвертация в pandas...
Размер данных: X_train=(750000, 2444), X_test=(250000, 2444)

Отбор признаков для каждого таргета:

[1/41] target_1_1...
  Отбор признаков для target_1_1...
    Подвыборка: 200000 строк (положительных: 7797.0)
    Отобрано 400 признаков

[2/41] target_1_2...
    Редкий класс (ratio=0.0034), subsample_size=200000
  Отбор признаков для target_1_2...
    Подвыборка: 200000 строк (положительных: 2569.0)
    Отобрано 400 признаков

[3/41] target_1_3...
  Отбор признаков для target_1_3...
    Подвыборка: 200000 строк (положительных: 17839.0)
    Отобрано 400 признаков

[4/41] target_1_4...
  Отбор признаков для target_1_4...
    Подвыборка: 200000 строк (положительных: 17572.0)
    Отобрано 400 признаков

[5/41] target_1_5...
    Редкий класс (ratio=0.0018), subsample_size=200000
  Отбор признаков для target_1_5...
    Подвыборка: 200000 строк (положительных: 1379.0)
    Отобрано 400 признаков

[6/41] target_2_1...
    Редкий класс (ratio=0.0071), subsample_size=20000

# 4. Функция обучения одного LightGBM
### Логика дейтсвий
Binary Relevance с кросс-валидацией: 
- Для каждого из 41 продукта строится своя модель LightGBM.
- Данные разбиваются на 3 фолда, на каждом фолде модель обучается на двух частях, валидируется на третьей.
- Собираются out‑of‑fold (OOF) предсказания для всех тренировочных клиентов, чтобы честно оценить Macro ROC‑AUC без переобучения.
- Тестовые предсказания получаются усреднением прогнозов трёх моделей.
- Итоговая метрика — Macro Averaged ROC‑AUC — вычисляется один раз по заполненной OOF‑матрице.

In [ ]:
N_FOLDS  = 3
def train_and_predict(params, model_name='LGB', return_models=False):
    """
    Обучает модели и возвращает:
    - oof: предсказания на обучающих данных
    - pred: предсказания на тестовых данных
    - auc: macro AUC на OOF
    - models: список обученных моделей (опционально)
    """
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    
    # матрица размером (750000,41)
    # куда будут записаны out‑of‑fold предсказания вероятности 
    # принадлежности к классу 1 для каждого клиента и каждого продукта.
    oof = np.zeros((len(X_train), len(target_cols)))
    # такая же матрица, но для тестовых клиентов (250000)
    pred = np.zeros((len(X_test), len(target_cols)))
    models_list = [] if return_models else None
    
    for t_idx, target_name in enumerate(target_cols):
        # Из словаря, созданного на этапе отбора признаков
        # берём имена TOP_K_FEATURES наиболее важных колонок именно для этого продукта.
        selected = selected_features_dict[target_name]
        # Из тренировочной и тестовой матриц оставляем только эти признаки.
        X_tr_sel = X_train[selected]
        X_te_sel = X_test[selected]
        # Формируем список категориальных признаков из тех, что попали в отобранные.
        cat_sel = [c for c in cat_feature_names if c in selected]
        # одномерный numpy‑массив истинных меток (0 или 1) для данного продукта.
        y_target = y_train[target_name].values
        
        fold_preds_test = []
        fold_models = []
        
        # На каждой итерации получаем tr — индексы объектов, попадающих в обучающую часть, 
        # и val — в валидационную. Их объединение — все индексы от 0 до N-1.
        for fold, (tr_idx, val_idx) in enumerate(kf.split(X_tr_sel)):
            X_tr, X_val = X_tr_sel.iloc[tr_idx], X_tr_sel.iloc[val_idx]
            y_tr, y_val = y_target[tr_idx], y_target[val_idx]
            
            model = lgb.LGBMClassifier(**params)
            # Валидационная часть используется для ранней остановки: 
            # если AUC на ней (LightGBM по умолчанию для бинарной классификации использует binary_logloss, 
            # но можно изменить метрику; здесь не указана, поэтому по умолчанию — logloss) 
            # не улучшается 50 итераций подряд, обучение прекращается.
            model.fit(
                X_tr, y_tr,
                eval_set=[(X_val, y_val)],
                callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)],
                categorical_feature=cat_sel,
            )
            
            # Для валидационных объектов сохраняются предсказанные вероятности положительного класса. 
            # Индексы val_idx соответствуют исходной нумерации клиентов — так мы заполняем OOF‑матрицу, 
            # не нарушая порядка.
            oof[val_idx, t_idx] = model.predict_proba(X_val)[:, 1]
            # Для всего тестового множества вычисляются вероятности с помощью только что обученной модели, 
            # и результат сохраняется в список. Модель сохраняется для возможного дальнейшего использования.
            fold_preds_test.append(model.predict_proba(X_te_sel)[:, 1])
            fold_models.append(model)
            
            del X_tr, X_val, y_tr, y_val
            gc.collect()
        
        # Тестовые вероятности для данного продукта усредняются по трём фолдам. 
        # Это даёт более стабильную и надёжную оценку, чем одна модель, и снижает дисперсию прогноза.
        pred[:, t_idx] = np.mean(fold_preds_test, axis=0)
        models_list.append(fold_models) if return_models else None
        
        # На основе истинных меток y_target и OOF‑предсказаний 
        # (уже заполненных для всех клиентов после трёх фолдов) 
        # считаем ROC‑AUC для одного продукта. 
        # Это показывает, насколько хорошо модель ранжирует клиентов по вероятности иметь этот продукт.
        auc = roc_auc_score(y_target, oof[:, t_idx])
        print(f'  [{t_idx+1:02d}/{len(target_cols)}] {target_name:15s} AUC = {auc:.4f}')
    
    # Вычисляется Macro Averaged ROC‑AUC в точности, как описано в правилах соревнования: 
    # усреднение ROC‑AUC по всем 41 классам. y_train.values — матрица истинных меток размера (750000, 41), 
    # oof — матрица предсказаний того же размера. 
    # Параметр average='macro' заставляет sklearn вычислить AUC для каждого столбца независимо и 
    # вернуть среднее арифметическое.
    macro_auc = roc_auc_score(y_train.values, oof, average='macro')
    print(f'\n=== {model_name} OOF Macro AUC = {macro_auc:.5f} ===')
    
    if return_models:
        return oof, pred, macro_auc, models_list
    return oof, pred, macro_auc

## 5. LightGBM-1 — широкие деревья, быстрый

In [28]:
params_fast = {
    **BASE_PARAMS,
    'n_estimators': 300,
    'learning_rate': 0.1,
    'num_leaves': 31,
    'min_child_samples': 50,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'random_state': SEED,
}

print("\nОбучаем быструю модель...")
oof_fast, pred_fast, auc_fast = train_and_predict(params_fast, 'LGB-Fast')

np.save(f'{OUT_DIR}oof_1.npy',  oof_fast)
np.save(f'{OUT_DIR}pred_1.npy', pred_fast)
print('Сохранено: oof_1.npy, pred_1.npy')


Обучаем быструю модель...
  [01/41] target_1_1      AUC = 0.9114
  [02/41] target_1_2      AUC = 0.8162
  [03/41] target_1_3      AUC = 0.8667
  [04/41] target_1_4      AUC = 0.8277
  [05/41] target_1_5      AUC = 0.8865
  [06/41] target_2_1      AUC = 0.8201
  [07/41] target_2_2      AUC = 0.9332
  [08/41] target_2_3      AUC = 0.7909
  [09/41] target_2_4      AUC = 0.7384
  [10/41] target_2_5      AUC = 0.7263
  [11/41] target_2_6      AUC = 0.7287
  [12/41] target_2_7      AUC = 0.7806
  [13/41] target_2_8      AUC = 0.8426
  [14/41] target_3_1      AUC = 0.6851
  [15/41] target_3_2      AUC = 0.9113
  [16/41] target_3_3      AUC = 0.7328
  [17/41] target_3_4      AUC = 0.9263
  [18/41] target_3_5      AUC = 0.9563
  [19/41] target_4_1      AUC = 0.8425
  [20/41] target_5_1      AUC = 0.7378
  [21/41] target_5_2      AUC = 0.6932
  [22/41] target_6_1      AUC = 0.7160
  [23/41] target_6_2      AUC = 0.7186
  [24/41] target_6_3      AUC = 0.7488
  [25/41] target_6_4      AUC = 0.844

## 6. LightGBM-2 — глубокие деревья, медленный learning rate

In [29]:
params_medium = {
    **BASE_PARAMS,
    'n_estimators': 800,
    'learning_rate': 0.05,
    'num_leaves': 63,
    'min_child_samples': 30,
    'subsample': 0.75,
    'colsample_bytree': 0.75,
    'reg_alpha': 0.08,
    'reg_lambda': 1.5,
    'random_state': SEED + 100,
}

print("\nОбучаем среднюю модель...")
oof_medium, pred_medium, auc_medium = train_and_predict(params_medium, 'LGB-Medium')

np.save(f'{OUT_DIR}oof_medium.npy',  oof_medium)
np.save(f'{OUT_DIR}pred_medium.npy', pred_medium)
print('Сохранено: oof_medium.npy, pred_medium.npy')


Обучаем среднюю модель...
  [01/41] target_1_1      AUC = 0.9142
  [02/41] target_1_2      AUC = 0.8215
  [03/41] target_1_3      AUC = 0.8695
  [04/41] target_1_4      AUC = 0.8304
  [05/41] target_1_5      AUC = 0.8943
  [06/41] target_2_1      AUC = 0.8213
  [07/41] target_2_2      AUC = 0.9352
  [08/41] target_2_3      AUC = 0.7899
  [09/41] target_2_4      AUC = 0.7397
  [10/41] target_2_5      AUC = 0.7388
  [11/41] target_2_6      AUC = 0.7298
  [12/41] target_2_7      AUC = 0.8222
  [13/41] target_2_8      AUC = 0.9736
  [14/41] target_3_1      AUC = 0.6888
  [15/41] target_3_2      AUC = 0.9127
  [16/41] target_3_3      AUC = 0.7443
  [17/41] target_3_4      AUC = 0.9299
  [18/41] target_3_5      AUC = 0.9655
  [19/41] target_4_1      AUC = 0.8370
  [20/41] target_5_1      AUC = 0.7346
  [21/41] target_5_2      AUC = 0.6732
  [22/41] target_6_1      AUC = 0.7109
  [23/41] target_6_2      AUC = 0.7128
  [24/41] target_6_3      AUC = 0.7457
  [25/41] target_6_4      AUC = 0.843

In [ ]:
params_slow = {
    **BASE_PARAMS,
    'n_estimators': 1500,
    'learning_rate': 0.03,
    'num_leaves': 127,
    'min_child_samples': 20,
    'subsample': 0.7,
    'colsample_bytree': 0.7,
    'reg_alpha': 0.05,
    'reg_lambda': 2.0,
    'random_state': SEED + 200,
}

print("\nОбучаем медленную модель...")
oof_slow, pred_slow, auc_slow = train_and_predict(params_slow, 'LGB-Slow')

np.save(f'{OUT_DIR}oof_slow.npy',  oof_slow)
np.save(f'{OUT_DIR}pred_slow.npy', pred_slow)
print('Сохранено: oof_slow.npy, pred_slow.npy')


Обучаем медленную модель...
  [01/41] target_1_1      AUC = 0.9178
  [02/41] target_1_2      AUC = 0.8284
  [03/41] target_1_3      AUC = 0.8726
  [04/41] target_1_4      AUC = 0.8343
  [05/41] target_1_5      AUC = 0.9015
  [06/41] target_2_1      AUC = 0.8249
  [07/41] target_2_2      AUC = 0.9370
  [08/41] target_2_3      AUC = 0.7853
  [09/41] target_2_4      AUC = 0.7495
  [10/41] target_2_5      AUC = 0.7317
  [11/41] target_2_6      AUC = 0.7405
  [12/41] target_2_7      AUC = 0.8416
  [13/41] target_2_8      AUC = 0.9847
  [14/41] target_3_1      AUC = 0.6940
  [15/41] target_3_2      AUC = 0.9141
  [16/41] target_3_3      AUC = 0.7160
  [17/41] target_3_4      AUC = 0.9324
  [18/41] target_3_5      AUC = 0.9703
  [19/41] target_4_1      AUC = 0.8441
  [20/41] target_5_1      AUC = 0.7333
  [21/41] target_5_2      AUC = 0.7120
  [22/41] target_6_1      AUC = 0.7194
  [23/41] target_6_2      AUC = 0.7222
  [24/41] target_6_3      AUC = 0.7537
  [25/41] target_6_4      AUC = 0.8

In [51]:
params_fm = {
    **BASE_PARAMS,
    'n_estimators': 500,
    'learning_rate': 0.07,
    'num_leaves': 45,
    'min_child_samples': 20,
    'subsample': 0.75,
    'colsample_bytree': 0.75,
    'reg_alpha': 0.05,
    'reg_lambda': 1.0,
    'random_state': SEED + 50,
}
     
print("\nОбучаем между быстрой и средней моделью...")
oof_fm, pred_fm, auc_fm = train_and_predict(params_fm, 'LGB-FAST-MEDIUM')

np.save(f'{OUT_DIR}oof_fm.npy',  oof_fm)
np.save(f'{OUT_DIR}pred_fm.npy', pred_fm)


Обучаем между быстрой и средней моделью...
  [01/41] target_1_1      AUC = 0.9140
  [02/41] target_1_2      AUC = 0.8192
  [03/41] target_1_3      AUC = 0.8698
  [04/41] target_1_4      AUC = 0.8313
  [05/41] target_1_5      AUC = 0.8953
  [06/41] target_2_1      AUC = 0.8229
  [07/41] target_2_2      AUC = 0.9346
  [08/41] target_2_3      AUC = 0.7950
  [09/41] target_2_4      AUC = 0.7428
  [10/41] target_2_5      AUC = 0.7327
  [11/41] target_2_6      AUC = 0.7306
  [12/41] target_2_7      AUC = 0.8414
  [13/41] target_2_8      AUC = 0.8860
  [14/41] target_3_1      AUC = 0.6873
  [15/41] target_3_2      AUC = 0.9124
  [16/41] target_3_3      AUC = 0.7278
  [17/41] target_3_4      AUC = 0.9083
  [18/41] target_3_5      AUC = 0.9627
  [19/41] target_4_1      AUC = 0.8380
  [20/41] target_5_1      AUC = 0.7373
  [21/41] target_5_2      AUC = 0.7003
  [22/41] target_6_1      AUC = 0.7176
  [23/41] target_6_2      AUC = 0.7179
  [24/41] target_6_3      AUC = 0.7464
  [25/41] target_6_4

## 7. Ансамбль

In [ ]:
all_oof = [oof_fast, oof_medium, oof_slow, oof_fm]
all_pred = [pred_fast, pred_medium, pred_slow, pred_fm]
all_auc = [auc_fast, auc_medium, auc_slow, auc_fm]

total_auc = sum(all_auc)
simple_weights = [auc / total_auc for auc in all_auc]

final_predictions = np.zeros_like(all_pred[0])
for i, pred in enumerate(all_pred):
    final_predictions += simple_weights[i] * pred

oof_ensemble = np.zeros_like(all_oof[0])
for i, oof in enumerate(all_oof):
    oof_ensemble += simple_weights[i] * oof
ensemble_auc = roc_auc_score(y_train.values, oof_ensemble, average='macro')

print(f"\nВеса по AUC (мгновенно):")
print(f"  Быстрая модель:   {simple_weights[0]:.4f} (AUC={auc_fast:.5f})")
print(f"  Средняя модель:   {simple_weights[1]:.4f} (AUC={auc_medium:.5f})")
print(f"  Жесткая модель:   {simple_weights[2]:.4f} (AUC={auc_slow:.5f})")
print(f"  Жесткая модель:   {simple_weights[3]:.4f} (AUC={auc_fm:.5f})")
print(f"\n  АНСАМБЛЬ OOF Macro AUC = {ensemble_auc:.5f}")


Веса по AUC (мгновенно):
  Быстрая модель:   0.2486 (AUC=0.81368)
  Средняя модель:   0.2505 (AUC=0.82003)
  Жесткая модель:   0.2512 (AUC=0.82238)
  Жесткая модель:   0.2497 (AUC=0.81719)

  АНСАМБЛЬ OOF Macro AUC = 0.82712


In [ ]:
def add_meta_features_to_predictions(oof_ensemble, final_predictions, target_cols, y_train, threshold=0.75):
    """
    Улучшает предсказания для сложных классов с помощью мета-фичей.
    Возвращает улучшенные предсказания.
    """
    print("\n" + "="*60)
    print("ДОБАВЛЕНИЕ МЕТА-ФИЧЕЙ ДЛЯ СЛОЖНЫХ КЛАССОВ")
    print("="*60)
    
    target_auc = {}
    for t_idx, target_name in enumerate(target_cols):
        target_auc[target_name] = roc_auc_score(y_train[target_name].values, oof_ensemble[:, t_idx])
    
    hard_classes = [name for name, auc in target_auc.items() if auc < threshold]
    print(f"Сложных классов (AUC < {threshold}): {len(hard_classes)}")
    
    if not hard_classes:
        print("Нет сложных классов, возвращаем исходные предсказания")
        return final_predictions
    
    target_corr = y_train[target_cols].corr()
    
    selected_features_dict = {}
    for t in target_cols:
        path = f'{SELECTED_DIR}/{t}.npy'
        if os.path.exists(path):
            selected_features_dict[t] = np.load(path).tolist()
    
    improved_predictions = final_predictions.copy()
    
    for target_name in hard_classes:
        print(f"\nУлучшаем {target_name} (AUC={target_auc[target_name]:.4f})...")
        
        selected = selected_features_dict.get(target_name, X_train.columns.tolist()[:500])
        X_train_base = X_train[selected].copy()
        X_test_base = X_test[selected].copy()
        
        corr_others = target_corr[target_name].drop(target_name)
        top_targets = corr_others.nlargest(10).index.tolist()
        
        for other in top_targets:
            other_idx = target_cols.index(other)
            X_train_base[f'meta_{other}'] = oof_ensemble[:, other_idx]
            X_test_base[f'meta_{other}'] = final_predictions[:, other_idx]
        
        y_target = y_train[target_name].values
        
        meta_params = {
            'objective': 'binary',
            'metric': 'auc',
            'n_estimators': 300,
            'learning_rate': 0.03,
            'num_leaves': 31,
            'min_child_samples': 30,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'reg_alpha': 0.1,
            'reg_lambda': 1.0,
            'n_jobs': -1,
            'random_state': SEED + 300,
            'verbose': -1,
            'device': 'gpu',
            'gpu_platform_id': 0,
            'gpu_device_id': 0,
        }
        
        kf = KFold(n_splits=3, shuffle=True, random_state=SEED)
        meta_test_preds = []
        
        for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train_base)):
            X_tr, X_val = X_train_base.iloc[tr_idx], X_train_base.iloc[val_idx]
            y_tr, y_val = y_target[tr_idx], y_target[val_idx]
            
            model = lgb.LGBMClassifier(**meta_params)
            model.fit(X_tr, y_tr)
            meta_test_preds.append(model.predict_proba(X_test_base)[:, 1])
            
            del model
            gc.collect()
        
        target_idx = target_cols.index(target_name)
        improved_predictions[:, target_idx] = np.mean(meta_test_preds, axis=0)
        print(f"  Готово")
    
    return improved_predictions

In [55]:
predict_columns = [f"predict_{col.replace('target_', '')}" for col in target_cols]

In [ ]:

improved_predictions = add_meta_features_to_predictions(
     oof_ensemble, final_predictions, target_cols, y_train, threshold=0.75
)

predictions_df = pl.DataFrame(improved_predictions, schema=predict_columns)
submission_meta = pl.DataFrame({'customer_id': test_customer_ids['customer_id']}).hstack(predictions_df)
submission_meta.write_parquet(f'{OUT_DIR}submission_meta3.parquet')
print(f"✅ Сабмит с мета-фичами сохранён: {OUT_DIR}submission_meta3.parquet")


ДОБАВЛЕНИЕ МЕТА-ФИЧЕЙ ДЛЯ СЛОЖНЫХ КЛАССОВ
Сложных классов (AUC < 0.75): 9

Улучшаем target_2_6 (AUC=0.7453)...
  Готово

Улучшаем target_3_1 (AUC=0.6921)...
  Готово

Улучшаем target_3_3 (AUC=0.7493)...
  Готово

Улучшаем target_5_1 (AUC=0.7414)...
  Готово

Улучшаем target_5_2 (AUC=0.7124)...
  Готово

Улучшаем target_6_1 (AUC=0.7233)...
  Готово

Улучшаем target_6_2 (AUC=0.7244)...
  Готово

Улучшаем target_9_3 (AUC=0.6764)...
  Готово

Улучшаем target_9_6 (AUC=0.6883)...
  Готово
✅ Сабмит с мета-фичами сохранён: data/submission_meta3.parquet


## 8. Формирование и проверка сабмита

In [ ]:
predict_columns = [f"predict_{col.replace('target_', '')}" for col in target_cols]

predictions_df = pl.DataFrame(final_predictions, schema=predict_columns)

submission = pl.DataFrame({'customer_id': test_customer_ids['customer_id']}).hstack(predictions_df)

submission.write_parquet(f'{OUT_DIR}submission3.parquet')
print(f"✅ Сабмит сохранён: {OUT_DIR}submission3.parquet")
print(f"   Размер: {submission.shape[0]} строк, {submission.shape[1]} колонок")
print("\nПервые 3 строки:")
print(submission.head(3))


✅ Сабмит сохранён: data/submission3.parquet
   Размер: 250000 строк, 42 колонок

Первые 3 строки:
shape: (3, 42)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ customer_ ┆ predict_1 ┆ predict_1 ┆ predict_1 ┆ … ┆ predict_9 ┆ predict_9 ┆ predict_9 ┆ predict_ │
│ id        ┆ _1        ┆ _2        ┆ _3        ┆   ┆ _6        ┆ _7        ┆ _8        ┆ 10_1     │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│ i32       ┆ f64       ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 1750001   ┆ 0.000814  ┆ 0.001903  ┆ 0.009748  ┆ … ┆ 0.404513  ┆ 0.095315  ┆ 0.000537  ┆ 0.347165 │
│ 1750002   ┆ 0.007246  ┆ 0.0041    ┆ 0.040597  ┆ … ┆ 0.326848  ┆ 0.098047  ┆ 0.000328  ┆ 0.235731 │
│ 1750003   ┆ 0.00161   ┆ 0.002152  ┆ 0.008864  ┆ … ┆ 0.278688  ┆ 0.041462  ┆ 0